In [1]:
%cd /home/parthgandhi/Projects/MLBot/mlbot/src

/home/parthgandhi/Projects/MLBot/mlbot/src


In [2]:
import polars as pl
import polars.selectors as cs
from ma_bands.features import add_indicators, detect_events

In [3]:
data = pl.scan_parquet("test_data.parquet")

ma_window_size = 50
atr_window_size = 14
atr_multi = 0.5
use_ema = False

In [7]:
res = add_indicators(
    data=data,
    ma_window_size=ma_window_size,
    use_ema=use_ema,
    atr_window_size=atr_window_size,
    atr_multi=atr_multi,
)

# res = detect_events(data=res)

In [10]:
SYMBOL_LIST = ["USHAMART", "QPOWER"]

In [14]:
res.filter(pl.col("symbol").is_in(SYMBOL_LIST)).with_columns(
    pl.col("pos")
    .shift(1)
    .over(partition_by="symbol", order_by="timestamp", descending=False)
    .alias("prev_pos")
).filter((pl.col("pos") == 0) & (pl.col("prev_pos") == 0)).collect()

symbol,timestamp,close,close_sma_50,atr_14,upper_band,lower_band,pos,prev_pos
str,date,f64,f64,f64,f64,f64,i32,i32
"""USHAMART""",2025-03-07,332.9,332.93,15.66,340.76,325.1,0,0
"""USHAMART""",2025-03-20,328.15,323.68,13.09,330.225,317.135,0,0
"""USHAMART""",2025-04-16,320.6,316.95,13.04,323.47,310.43,0,0
"""USHAMART""",2025-04-17,311.15,316.33,12.72,322.69,309.97,0,0
"""USHAMART""",2025-04-21,312.7,315.77,11.88,321.71,309.83,0,0
…,…,…,…,…,…,…,…,…
"""USHAMART""",2026-04-07,405.5,411.64,15.4,419.34,403.94,0,0
"""QPOWER""",2026-01-07,811.7,792.57,40.14,812.64,772.5,0,0
"""QPOWER""",2026-01-08,792.15,788.73,40.31,808.885,768.575,0,0


In [6]:
res.with_columns(pl.col("timestamp").dt.year().alias("year")).group_by(
    "year", "is_bounce"
).len().with_columns(
    (pl.col("len") * 100 / pl.col("len").sum()).over(partition_by="year").alias("pct"),
    pl.col("len").sum().over(partition_by="year").alias("total_count"),
).sort(["year", "is_bounce"], descending=[False, True]).collect()

year,is_bounce,len,pct,total_count
i32,bool,u32,f64,u32
2025,true,18923,54.963983,34428
2025,false,15505,45.036017,34428
2026,true,6271,52.38493,11971
2026,false,5700,47.61507,11971
